# Generate SAT Instance
* Including SAT, MaxSAT
* The SAT Generator will generate the sat problem instance, and also given the corresponding answer to the generated instance, instead of The MaxSAT Generator only generating the problem instance.

In [ ]:
"""
    You must run this section of codes before any other section
"""

from typing import *
from enum import Enum
from abc import ABC, abstractmethod
import os
import math
import time
import random

# ============================== Required Constants ============================== #
class StatusOfSAT(Enum):
  SAT = "SAT"
  UNSAT = "UNSAT"
# ============================== Required Constants ============================== #


# ============================== Abstract Class ============================== #
class AbsSATGenerator(ABC):
  @abstractmethod
  def generate(self) -> bool: pass

  @abstractmethod
  def get_clauses(self) -> List[List[int]]: pass

class AbsAnswerFetcher(ABC):
  @abstractmethod
  def get_assignments(self) -> List[bool]: pass

  @abstractmethod
  def get_answer_of_clauses(self) -> Tuple[List[bool], int, StatusOfSAT]: pass

class AbsEnsureSATGenerator(ABC):
  @abstractmethod
  def ensure_generate(self) -> bool: pass
# ============================== Abstract Class ============================== #


# ============================== Instance Data Structure ============================== #
class Instance():
  def __init__(self, clauses: List[List[int]]):
    if not clauses or len(clauses) == 0:
      raise ValueError("The given clauses are empty which is not allowed")

    clause_size = len(clauses[0])
    for clause in clauses:
      if len(clause) != clause_size:
        raise ValueError("The given clauses are not all the same size")

    self.clauses: List[List[int]] = clauses

  def __getitem__(self, index: int) -> List[int]:
    return self.clauses[index]

  def __len__(self) -> int:
    return len(self.clauses)
# ============================== Instance Data Structure ============================== #

In [ ]:
# ============================== K-SAT Instance Generator ============================== #
class KSATInstanceGenerator(AbsSATGenerator, AbsAnswerFetcher, AbsEnsureSATGenerator):
  """
      # K-SAT Instance Generator
      ## Class to generate K-SAT Instance with True of False to each clauses.
      ==========================================================================================
      - Inputs:
        * ```clause_size: int``` : the size of each clause
        * ```max_num_of_variables: int``` : the maximum number of variables in each clause
        * ```max_num_of_clauses: int``` : the maximum number of clauses
        * ```is_random: bool``` :
          if this argument is true,
          the ```max_num_of_variables``` and ```max_num_of_clauses``` will be the number of variables and the number of clauses directly,
          else if it is false, the number of variables and the number of clauses will be random.
          (The Range of the number of variables and the number of clauses are ```[max_num_of_variables / 2, max_num_of_variables]```, ```[max_num_of_clauses / 2, max_num_of_clauses]```)

      - Notes:
        * Ignore the index 0 of assignments, since if there's x_0, the clauses cannot represent the negative properly

      - Examples:
        * ```self.clauses = [[3, -2, 1], [2, 3, 1], [2, 1, -3]]``` stands for:
          [[x_3, ¬x_2, x_1], [x_2, x_3, x_1], [x_2, x_1, ¬x_3]]
        * ```self.assignments = [false, true, false, true]``` stands for:
          x_1 = true, x_1 = false, x_2 = true
        * ```self.answer_of_clauses = [true, true, false]``` stands for:
          self.clauses[0] = true, self.clauses[1] = true, self.clauses[2] = false
        * ```self.sat_count = 2``` stands for:
          there's totally two clauses that are satisfied

      - Requirements:
        * ```StatusOfSAT``` enum as the return type of get_answer_of_clauses()
  """
  def __init__(
      self,
      clause_size: int = 3,
      max_num_of_variables: int = 100,
      max_num_of_clauses: int = 1000,
      is_random: bool = True
    ):
    self.clause_size: int = clause_size
    self.num_of_variables: int = max(1, random.randint(max_num_of_variables / 2, max_num_of_variables)) if is_random else max_num_of_variables
    self.num_of_clauses: int = max(1, random.randint(max_num_of_clauses / 2, max_num_of_clauses)) if is_random else max_num_of_clauses
    self.assignments: List[bool] = []
    self.true_assignments: Set[int] = set()
    self.clauses: List[List[int]] = []
    self.answer_of_clauses: List[bool] = []
    self.sat_count: int = 0

  def generate(self) -> bool:
    """
      ## Method to generate a SAT instance

      - Generate stuff included:
        * assignments
        * true_assignments
        * clauses
        * answer_of_clauses
        * sat_count
    """
    try:
      self.assignments = [False] * (self.num_of_variables + 1)
      self.true_assignments.clear()
      self.clauses.clear()
      self.answer_of_clauses = [False] * self.num_of_clauses
      self.sat_count = 0

      for i in range(self.num_of_variables + 1):
        self.assignments[i] = random.choice([True, False])
        if self.assignments[i]:
          self.true_assignments.add(i)

      for i in range(self.num_of_clauses):
        current_clause: List[int] = [-1] * self.clause_size
        random_variables = random.sample(range(1, self.num_of_variables + 1), self.clause_size + 1)
        normal_variables, ensure_variable = random_variables[:self.clause_size], random_variables[self.clause_size]

        for j, variable in enumerate(normal_variables):
          selected_sign = random.choice([True, False])
          current_clause[j] = variable if selected_sign else -variable
          if selected_sign == self.assignments[variable]:
            self.answer_of_clauses[i] = True

        self.sat_count += (1 if self.answer_of_clauses[i] is True else 0)
        self.clauses.append(current_clause)

      return True
    except Exception as error:
      print(error)
      return False

  def ensure_generate(self) -> bool:
    """
      ## Method to generate an ensured SAT instance of MaxSAT

      - Generate stuff included:
        * assignments
        * true_assignments
        * clauses
        * answer_of_clauses
        * sat_count
    """
    try:
      self.assignments = [False] * (self.num_of_variables + 1)
      self.true_assignments.clear()
      self.clauses.clear()
      self.answer_of_clauses = [True] * self.num_of_clauses
      self.sat_count = self.num_of_clauses

      for i in range(self.num_of_variables + 1):
        self.assignments[i] = random.choice([True, False])
        if self.assignments[i]:
          self.true_assignments.add(i)

      for i in range(self.num_of_clauses):
        current_clause: List[int] = [-1] * self.clause_size
        random_variables = random.sample(range(1, self.num_of_variables + 1), self.clause_size + 1)
        normal_variables, ensure_variable = random_variables[:self.clause_size], random_variables[self.clause_size]

        for j, variable in enumerate(normal_variables):
          selected_sign = random.choice([True, False])
          current_clause[j] = variable if selected_sign else -variable
          if selected_sign == self.assignments[variable]:
            self.answer_of_clauses[i] = True

        # if there's no true variable included, then insert a random true assignment forcibly
        if self.answer_of_clauses[i] is not True:
          self.current_clause[random.randint(0, self.clause_size - 1)] = ensure_variable if self.assignments[ensure_variable] else -ensure_variable
        self.clauses.append(current_clause)

      return True
    except Exception as error:
      print(error)
      return False

  def get_clauses(self) -> List[List[int]]:
    """
      ## Method to get the clauses of MaxSAT

      Notes: Make sure the generate() or generate_ensure_sat() has been run
    """
    return self.clauses

  def get_assignments(self) -> List[bool]:
    """
      ## Method to get the assignments of MaxSAT

      Notes: Make sure the generate() or generate_ensure_sat() has been run
    """
    return self.assignments

  def get_answer_of_clauses(self) -> Tuple[List[bool], int, StatusOfSAT]:
    """
      ## Method to get the answer to each clauses, sat count, and the entire result of MaxSAT

      Notes: Make sure the generate() or generate_ensure_sat() has been run
    """
    return self.answer_of_clauses, self.sat_count, (StatusOfSAT.SAT if self.sat_count == self.num_of_clauses else StatusOfSAT.UNSAT)
# ============================== K-SAT Instance Generator ============================== #

In [ ]:
generator = KSATInstanceGenerator(clause_size=3, max_num_of_clauses=10, max_num_of_variables=5, is_random=False)
generator.generate()
clauses = generator.get_clauses()
assignments = generator.get_assignments()
answer_of_clauses, sat_count, status = generator.get_answer_of_clauses()
print("============================== Result ==============================")
print(clauses)
print(assignments)
print(answer_of_clauses)
print(f"Number of the Satisfied Clauses: {sat_count} / {len(clauses)}")
print("Is " + ("SAT" if status == StatusOfSAT.SAT else "UNSAT"))
print("============================== Result ==============================")

============================== Result ==============================
[[-4, -5, -1], [5, 1, 3], [-3, -4, 5], [2, -1, 3], [-5, -1, -3], [-5, 2, 1], [3, 4, -2], [-3, -4, -2], [-1, 3, -4], [3, -1, 4]]
[False, False, False, True, True, False]
[True, True, False, True, True, True, True, True, True, True]
Number of the Satisfied Clauses: 9 / 10
Is UNSAT


In [ ]:
# ============================== Max-K-SAT Instance Generator ============================== #
class MaxSATInstanceGenerator(AbsSATGenerator):
  def __init__(
      self,
      clause_size: int = 3,
      max_num_of_variables: int = 100,
      max_num_of_clauses: int = 1000,
      is_random: bool = True
  ):
    self.clause_size: int = clause_size
    self.num_of_variables: int = max(1, random.randint(max_num_of_variables / 2, max_num_of_variables)) if is_random else max_num_of_variables
    self.num_of_clauses: int = max(1, random.randint(max_num_of_clauses / 2, max_num_of_clauses)) if is_random else max_num_of_clauses
    self.clauses: List[List[int]] = []

  def generate(self) -> bool:
    try:
      self.clauses.clear()

      for _ in range(self.num_of_clauses):
        current_clause: List[int] = [-1] * self.clause_size
        random_variables = random.sample(range(1, self.num_of_variables + 1), self.clause_size)
        variables = random_variables[:self.clause_size]

        for j, variable in enumerate(variables):
          selected_sign = random.choice([True, False])
          current_clause[j] = variable if selected_sign else -variable

        self.clauses.append(current_clause)

      return True
    except Exception as error:
      print(error)
      return False


  def get_clauses(self) -> List[List[int]]:
    return self.clauses;
# ============================== Max-K-SAT Instance Generator ============================== #

In [ ]:
generator = MaxSATInstanceGenerator(clause_size=3, max_num_of_clauses=100, max_num_of_variables=24, is_random=False)
generator.generate()
clauses = generator.get_clauses()
print(clauses)

[[9, -2, -4], [22, 1, -23], [-16, -2, 11], [19, -7, 22], [12, -21, -1], [15, 19, 5], [-17, 6, -14], [-20, -13, 15], [-10, -9, -23], [-19, -24, -12], [24, -17, 10], [-19, 14, 6], [10, -23, -22], [-7, 8, -10], [-13, 18, -14], [-3, 9, -17], [-17, 21, 24], [-6, 15, 18], [-24, 14, 8], [19, -3, 7], [-21, 7, -6], [12, 1, -23], [18, 2, -1], [22, -24, 16], [-18, 19, -16], [-15, -13, 6], [-8, 3, -21], [8, 16, -13], [-12, 14, 20], [20, 8, -12], [22, -10, -20], [-7, -20, -4], [-9, -16, 6], [14, -3, 24], [-6, -8, -19], [-6, 8, -19], [-1, -15, -14], [-11, 4, -9], [-15, -11, 24], [-12, 19, 11], [-17, -19, 22], [22, 9, 10], [-16, -19, 17], [6, 7, -2], [-24, -15, -9], [-10, 16, 22], [-19, -9, 15], [2, -11, -7], [-2, 24, -4], [-24, -1, 21], [7, 3, -2], [-2, 13, -4], [6, -17, -8], [-24, 13, 10], [-1, 4, 22], [-20, -23, 4], [14, -3, 2], [-3, 13, -23], [20, -23, -14], [-6, 12, 14], [6, -11, 10], [17, 7, 8], [12, -19, -4], [9, 17, 22], [17, 15, 10], [-15, 10, 16], [-24, 22, 8], [3, 14, 4], [13, 11, 23], [2,

In [ ]:
# ============================== Max-K-SAT Basic Solver ============================== #
class MaxSATBasicSolver(AbsAnswerFetcher):
  """
     ## This solver try all the possibles with exponential time complexity
  """
  def __init__(self, clauses: List[List[int]]):
    self.clauses: List[List[int]] = clauses
    self.num_of_variables: int = max(map(lambda clause: max(map(abs, clause)), clauses))
    self.num_of_clauses: int = len(clauses)
    self.clause_size: int = len(clauses[0])
    self.assignments: List[bool] = [False] * (self.num_of_variables + 1)
    self.active_variables: List[int] = []
    self.answer_of_clauses: List[bool] = []
    self.sat_count: int = 0

  def reduce_variable_space(self):
    seen: Set[int] = set()
    for clause in self.clauses:
      for variable in clause:
        if (not seen.__contains__(-variable)):
          seen.add(variable)
          self.active_variables.append(abs(variable))

  def _backtrack(
      self,
      i: int,
      current_assignments: List[bool],
      assign: bool,
    ):
    if (i >= len(self.active_variables)):
      return

    current_assignments[self.active_variables[i]] = assign

    current_sat_count = 0
    for clause in self.clauses:
      is_sat_clause = False
      for variable in clause:
        if (variable < 0 and not current_assignments[abs(variable)]) or (variable > 0 and current_assignments[abs(variable)]):
          is_sat_clause = True
          break
      if (is_sat_clause):
        current_sat_count += 1

    if (current_sat_count > self.sat_count):
      self.assignments = current_assignments.copy()
      self.sat_count = current_sat_count

    self._backtrack(i + 1, current_assignments, True)
    self._backtrack(i + 1, current_assignments, False)

  def solve(self) -> bool:
    try:
      self.reduce_variable_space()
      init_assignments = self.assignments.copy()
      self._backtrack(1, init_assignments, True)
      self._backtrack(1, init_assignments, False)
      for clause in self.clauses:
        for variable in clause:
          if (variable < 0 and not self.assignments[abs(variable)]) or (variable >= 0 and self.assignments[abs(variable)]):
            self.answer_of_clauses.append(True)
            break

      return True
    except Exception as error:
      print(error)
      return False


  def get_assignments(self) -> List[bool]:
    return self.assignments

  def get_answer_of_clauses(self) -> Tuple[List[bool], int, StatusOfSAT]:
     return self.answer_of_clauses, self.sat_count, (StatusOfSAT.SAT if self.sat_count == self.num_of_clauses else StatusOfSAT.UNSAT)
# ============================== Max-K-SAT Basic Solver ============================== #

In [ ]:
solver = MaxSATBasicSolver(clauses)
solver.solve()
print(solver.get_assignments())
print(solver.get_answer_of_clauses())

[[-3, -1, 2], [-3, -1, 2], [-1, -2, -3], [-2, 3, 1], [3, -2, 1], [-3, -1, 2], [-2, -3, 1], [2, -1, 3], [-1, 3, -2], [-2, -1, 3], [2, 3, 1], [-2, 1, -3], [3, -1, -2], [-2, 1, 3]]
[False, False, False, True]
([True, True, True, True, True, True, True, True, True, True, True, True, True, True], 14, <StatusOfSAT.SAT: 'SAT'>)


# Prepare Unweighted MaxSAT Dataset
* Download the dataset from MaxSAT Evalutation Benchmarks : https://maxsat-evaluations.github.io/2024/benchmarks.html

In [ ]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   38G   71G  35% /
tmpfs            64M     0   64M   0% /dev
shm             5.8G     0  5.8G   0% /dev/shm
/dev/root       2.0G  1.2G  820M  59% /usr/sbin/docker-init
/dev/sda1        85G   66G   20G  78% /kaggle/input
tmpfs           6.4G  120K  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [ ]:
# download the target .zip file to colab local storage
!wget -O /content/exact-unweighted.zip https://www.cs.helsinki.fi/group/coreo/MSE2024-instances/mse24-exact-unweighted.zip

--2025-04-28 06:32:42--  https://www.cs.helsinki.fi/group/coreo/MSE2024-instances/mse24-exact-unweighted.zip
Resolving www.cs.helsinki.fi (www.cs.helsinki.fi)... 128.214.166.78
Connecting to www.cs.helsinki.fi (www.cs.helsinki.fi)|128.214.166.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2075486968 (1.9G) [application/zip]
Saving to: ‘/content/exact-unweighted.zip’

/content/exact-unwe 100%[===================>]   1.93G  24.9MB/s    in 80s     

2025-04-28 06:34:04 (24.8 MB/s) - ‘/content/exact-unweighted.zip’ saved [2075486968/2075486968]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import zipfile
import os

zip_path = "/content/exact-unweighted.zip"
extract_path = "/content/unweighted_maxsat_data"

# try to unzip the .zip file
try:
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
      zip_ref.extractall(extract_path)
  print("Unzip successfully!")
except Exception as error:
  print(error)

[Errno 2] No such file or directory: '/content/exact-unweighted.zip'


In [ ]:
import glob

wcnf_files = glob.glob("/content/unweighted_maxsat_data/**/*.wcnf.xz", recursive=True)
print(f"There's {len(wcnf_files)} .wcnf files totally")

for file in wcnf_files[:5]:
    print(file)

There's 0 .wcnf files totally


## Check the inner by lzma

In [ ]:
import lzma

with lzma.open('/content/unweighted_maxsat_data/MaxSATQueriesinInterpretableClassifiers-adult_test_9_CNF_2_1.wcnf.xz', 'rt') as f:
    for _ in range(10):
        print(f.readline())

FileNotFoundError: [Errno 2] No such file or directory: '/content/unweighted_maxsat_data/MaxSATQueriesinInterpretableClassifiers-adult_test_9_CNF_2_1.wcnf.xz'

## Build WCNF Reader
* This WCNF reader function is used to read clauses as WCNF form in .wcnf.xz files
* Given a `filepath` in string type indicate the path to a wcnf.xz file as a parameter
* Return two clauses of `soft_clauses` and `hard_clauses` in order, note that each of them could be `0` as there're no soft or hard clauses in the file

In [ ]:
from typing import List, Tuple
import lzma

class WCNFReader():
  def __init__(self, mode='rt', encoding='utf-8', error_mode='ignore'):
    self.mode = mode
    self.encoding = encoding
    self.error_mode = error_mode
    self.content_comment_sign = 'c'
    self.content_ignore_sign = 'p'
    self.content_hard_sign = 'h'
    self.content_end_of_line_sign = '0'

  def read_wcnf_to_clauses(self, filepath: str) -> Tuple[List[List[int]], List[List[int]]]:
      soft_clauses = []
      hard_clauses = []
      with lzma.open(filepath, mode=self.mode, encoding=self.encoding, errors=self.error_mode) as file:
          for line in file:
              line = line.strip()
              if not line or line.startswith(self.content_comment_sign) or line.startswith(self.content_ignore_sign):
                  continue

              parts = line.split()

              if parts[-1] != self.content_end_of_line_sign:  # note that 0 indicate the end of line
                  continue

              # start with n (n for some integer) means the weighted of that clause,
              #            h means the current line(clause) is the hard clause,
              #            c means the current line is a comment
              try:
                  if parts[0] == self.content_hard_sign:
                      clause = list(map(int, parts[1:-1]))
                      hard_clauses.append(clause)
                  else:
                      weight = int(parts[0])
                      clause = list(map(int, parts[1:-1]))
                      if weight == 1:
                          soft_clauses.append(clause)
                      else:
                          hard_clauses.append(clause)
              except ValueError:
                  continue

      return soft_clauses, hard_clauses

### Try to read one random wcnf.xz file
* Also printing out the soft clauses and hard clauses inside it

In [ ]:
wcnf_reader = WCNFReader()

soft_clauses, hard_clauses = wcnf_reader.read_wcnf_to_clauses('/content/unweighted_maxsat_data/MaxSATQueriesinInterpretableClassifiers-adult_test_9_CNF_2_1.wcnf.xz')

print(f"Read {len(soft_clauses)} soft clauses")
print("Some of the read soft clauses : ", soft_clauses[-15:-1])

print(f"Read {len(hard_clauses)} hard clauses")
print("Some of the read hard clauses : ", hard_clauses[-15:-1])

Read 3544 soft clauses
Some of the read soft clauses :  [[-3530], [-3531], [-3532], [-3533], [-3534], [-3535], [-3536], [-3537], [-3538], [-3539], [-3540], [-3541], [-3542], [-3543]]
Read 63656 hard clauses
Some of the read hard clauses :  [[-8487, -83], [-8487, -98], [-8487, -139], [-8488, -149], [-8488, -157], [-8488, -174], [-8488, -182], [-8488, -197], [-8488, -200], [-8488, -206], [-8488, -211], [-8488, -212], [-8488, -227], [-8488, -242]]


# Setup PySAT Related and Solvers
* Including WCNF Formula, Examples, and Solvers
  - The Formula is suitable with the form of SAT we defined above.
  - The Examples are actually a converter to convert SAT instance(clauses form) to some industrial problems.
  - The Solvers include RC2, FN, LSU, etc.

## Build WCNF Converter
* For converting clauses with typing list to PySAT WCNF

In [ ]:
try:
  from pysat.formula import WCNF
except:
  !pip install python-sat
  from pysat.formula import WCNF

class WCNFConverter():
  def __init__(self):
    pass

  def convert_to_wcnf(self, clauses: List[List[int]]) -> WCNF:
    wcnf = WCNF()
    for clause in clauses:
      wcnf.append(clause)
    return wcnf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 24.2 MB/s eta 0:00:00


In [ ]:
wcnf_converter = WCNFConverter()
soft_clause_formula, hard_clause_formula \
  = wcnf_converter.convert_to_wcnf(soft_clauses), wcnf_converter.convert_to_wcnf(hard_clauses)
print("Soft Clause in WCNF :", soft_clause_formula)
print("Hard Clause in WCNF :", hard_clause_formula)

Soft Clause in WCNF : WCNF(from_string='p wcnf 3544 3544 1\n1 -1 0\n1 -2 0\n1 -3 0\n1 -4 0\n1 -5 0\n1 -6 0\n1 -7 0\n1 -8 0\n1 -9 0\n1 -10 0\n1 -11 0\n1 -12 0\n1 -13 0\n1 -14 0\n1 -15 0\n1 -16 0\n1 -17 0\n1 -18 0\n1 -19 0\n1 -20 0\n1 -21 0\n1 -22 0\n1 -23 0\n1 -24 0\n1 -25 0\n1 -26 0\n1 -27 0\n1 -28 0\n1 -29 0\n1 -30 0\n1 -31 0\n1 -32 0\n1 -33 0\n1 -34 0\n1 -35 0\n1 -36 0\n1 -37 0\n1 -38 0\n1 -39 0\n1 -40 0\n1 -41 0\n1 -42 0\n1 -43 0\n1 -44 0\n1 -45 0\n1 -46 0\n1 -47 0\n1 -48 0\n1 -49 0\n1 -50 0\n1 -51 0\n1 -52 0\n1 -53 0\n1 -54 0\n1 -55 0\n1 -56 0\n1 -57 0\n1 -58 0\n1 -59 0\n1 -60 0\n1 -61 0\n1 -62 0\n1 -63 0\n1 -64 0\n1 -65 0\n1 -66 0\n1 -67 0\n1 -68 0\n1 -69 0\n1 -70 0\n1 -71 0\n1 -72 0\n1 -73 0\n1 -74 0\n1 -75 0\n1 -76 0\n1 -77 0\n1 -78 0\n1 -79 0\n1 -80 0\n1 -81 0\n1 -82 0\n1 -83 0\n1 -84 0\n1 -85 0\n1 -86 0\n1 -87 0\n1 -88 0\n1 -89 0\n1 -90 0\n1 -91 0\n1 -92 0\n1 -93 0\n1 -94 0\n1 -95 0\n1 -96 0\n1 -97 0\n1 -98 0\n1 -99 0\n1 -100 0\n1 -101 0\n1 -102 0\n1 -103 0\n1 -104 0\n1 -105 0

## Build Brute Force SAT Solvers
* Same concept as the above MaxSATBasicSolver to try all the possible variable assigments. However, in each try we use the oracle to check sat or unsat in pysat.solvers, instead of iterating the clauses to get the number of satisfied clauses

In [ ]:
from pysat.formula import WCNF

# ============================== Brute Force Solver ============================== #
class BruteForceSolver(AbsAnswerFetcher):
  """
     ## This solver try all the possibles with exponential time complexity
  """
  def __init__(self, formula: WCNF):
    self.formula: WCNF = formula
    self.clauses = formula.hard + [cl for cl, _ in formula.soft]
    self.num_of_variables: int = max(map(lambda clause: max(map(abs, clause)), clauses))
    self.num_of_clauses: int = len(clauses)
    self.assignments: List[bool] = [False] * (self.num_of_variables + 1)
    self.active_variables: List[int] = []
    self.answer_of_clauses: List[bool] = []
    self.sat_count: int = 0

  def reduce_variable_space(self):
    seen: Set[int] = set()
    for clause in self.clauses:
      for variable in clause:
        if (not seen.__contains__(-variable)):
          seen.add(variable)
          self.active_variables = sorted(set(abs(lit) for clause in self.clauses for lit in clause if abs(lit) > 0))

  def _backtrack(
      self,
      i: int,
      current_assignments: List[bool],
      assign: bool,
  ):
    if i == len(self.active_variables):
      current_sat_count = 0
      for clause in self.clauses:
        is_sat_clause = False
        for variable in clause:
          if (variable < 0 and not current_assignments[abs(variable)]) \
          or (variable > 0 and current_assignments[abs(variable)]):
              is_sat_clause = True
              break
        if is_sat_clause:
          current_sat_count += 1

      if current_sat_count > self.sat_count:
        self.assignments[:] = current_assignments
        self.sat_count = current_sat_count
      return

    current_assignments[self.active_variables[i]] = assign

    self._backtrack(i + 1, current_assignments[:], True)
    self._backtrack(i + 1, current_assignments[:], False)

  def solve(self) -> bool:
    self.reduce_variable_space()
    init_assignments = self.assignments.copy()
    self._backtrack(
      1, 
      init_assignments, 
      True, 
    )
    self._backtrack(
      1, 
      init_assignments, 
      False, 
    )
    for clause in self.clauses:
      for variable in clause:
        if (variable < 0 and not self.assignments[abs(variable)]) \
        or (variable >= 0 and self.assignments[abs(variable)]):
          self.answer_of_clauses.append(True)
          break


  def get_assignments(self) -> List[bool]:
    return self.assignments

  def get_answer_of_clauses(self) -> Tuple[List[bool], int, StatusOfSAT]:
     return self.answer_of_clauses, self.sat_count, (StatusOfSAT.SAT if self.sat_count == self.num_of_clauses else StatusOfSAT.UNSAT)
# ============================== Brute Force Solver ============================== #

In [ ]:
solver = BruteForceSolver(convert_to_wcnf(clauses))
solver.solve()
print(solver.get_assignments())
print(solver.get_answer_of_clauses())

[False, False, True, False, False, False, True, True, True, False, False, True, False, True, True, True, True, True, False, False, True, False, True, False, True]
([True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True], 99, <StatusOfSAT.UNSAT: 'UNSAT'>)


## Apply Dataset to FM  Algorithm

In [ ]:
try:
  from pysat.examples.fm import FM
except Exception as error:
  !pip install python-sat
  from pysat.examples.fm import fm

start_time = time.time()

fm_solver = FM(soft_clause_formula)
solution = fm_solver.compute()

end_time = time.time()

if solution is not None:
  print("Best Assignment:", solution)
  print(f"Execution Time : {(end_time - start_time):.4f} seconds")
else:
  print("Unsatisfiable")

Best Assignment: True
Execution Time : 0.0092 seconds


## Apply Dataset to RC2 Algorithm

In [ ]:
try:
  from pysat.examples.rc2 import RC2
except Exception as error:
  !pip install python-sat
  from pysat.examples.rc2 import RC2

start_time = time.time()

rc2_solver = RC2(soft_clause_formula)
solution = rc2_solver.compute()

end_time = time.time()

if solution is not None:
  print("Best Assignment:", solution)
  print(f"Execution Time : {(end_time - start_time):.4f} seconds")
else:
  print("Unsatisfiable")

Best Assignment: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31, -32, -33, -34, -35, -36, -37, -38, -39, -40, -41, -42, -43, -44, -45, -46, -47, -48, -49, -50, -51, -52, -53, -54, -55, -56, -57, -58, -59, -60, -61, -62, -63, -64, -65, -66, -67, -68, -69, -70, -71, -72, -73, -74, -75, -76, -77, -78, -79, -80, -81, -82, -83, -84, -85, -86, -87, -88, -89, -90, -91, -92, -93, -94, -95, -96, -97, -98, -99, -100, -101, -102, -103, -104, -105, -106, -107, -108, -109, -110, -111, -112, -113, -114, -115, -116, -117, -118, -119, -120, -121, -122, -123, -124, -125, -126, -127, -128, -129, -130, -131, -132, -133, -134, -135, -136, -137, -138, -139, -140, -141, -142, -143, -144, -145, -146, -147, -148, -149, -150, -151, -152, -153, -154, -155, -156, -157, -158, -159, -160, -161, -162, -163, -164, -165, -166, -167, -168, -169, -170, -171, -172, -173, -174, -175, -176, -177, -178, -179, -180, -181, -182

## Apply Dataset to LSU Algorithm

In [ ]:
# wcnf = WCNF()
# wcnf.append([1], weight=1)   #  x1
# wcnf.append([-1], weight=2)  #  NOT x1
# wcnf.append([2, 3], weight=3) #  x2 OR x3
# wcnf.append([-2, -3], weight=4) #  NOT x2 OR NOT x3
# wcnf.append([-1, 2], weight=5) #  NOT x1 OR x2

try:
  import threading
  from pysat.examples.lsu import LSU
  from pysat.formula import WCNF
except Exception as error:
  !pip install python-sat
  import threading
  from pysat.examples.lsu import LSU
  from pysat.formula import WCNF

start_time = time.time()

lsu = LSU(soft_clause_formula, verbose=1)
is_sat = lsu.solve()

end_time = time.time()

if is_sat:
  print("Is Optimum Founded ?", lsu.found_optimum())
  print("Best Assignments : ", lsu.get_model())
  print(f"Execution Time : {(end_time - start_time):.4f} seconds")
else:
  print("Unsatisfiable")

o 0
s OPTIMUM FOUND
Is Optimum Founded ? True
Best Assignments :  <filter object at 0x795abcb93eb0>
Execution Time : 0.0042 seconds


## Build a Test Step for the Entire Test Procedure
* Since we need to read all the wcnf in all the fetched dataset which are seperated into couple files in the folder we specified, we need to build a reader function to read all of them so that we can test the models or algorithms using all the data (or some of the data).
* Instead of reading all the data once, and test the models or algorithms once, we iterate all the files we want inside a folder, for each model or algorithm:
  1. Fetch the clauses in a wcnf.xz file and formalize it to WCNF data structure
  2. Start the timer to calculate the time taken by the testing model or the algorithm
  3. Pass the clauses to the testing model or algorithm
  4. End the timer and accumulate the time it used to somewhere
  5. If there're any other files we want to include into the test, then go to the first step

In [ ]:
import time
import glob
import random
from enum import Enum
from pysat.formula import WCNF
from pysat.examples.fm import FM
from pysat.examples.rc2 import RC2
from pysat.examples.lsu import LSU

class SolverType(Enum):
  FM = "FM"
  RC2 = "RC2"
  LSU = "LSU"
  BruteForce = "BruteForce"

class TestStep():
  def __init__(
      self,
      fetch_count: int = -1,
      dataset_folder_path: str = "/content",
      fetch_random_file: bool = False,
      limit_solvers: List[SolverType] = [],
      verbose: int = 0,
    ):
    self.wcnf_reader: WCNFReader = WCNFReader()
    self.wcnf_converter: WCNFConverter = WCNFConverter()

    self.fetch_count: int = fetch_count
    self.dataset_folder_path: str = dataset_folder_path
    self.fetch_random_file: bool = fetch_random_file
    self.limit_solvers: List[SolverType] = limit_solvers
    self.verbose = verbose

    self.wcnf_file_paths: List[str] = []
    self._file_pointer: int = 0
    self._current_soft_clauses: List[List[int]] = []
    self._current_hard_clauses: List[List[int]] = []
    self.used_time: int = 0
    self._current_start_time: int = 0
    self._timer_lock = False
    self._file_pattern = '*.wcnf.xz'

    self.__init_fetch_all_wcnf_files_in_given_folder()

  def __init_fetch_all_wcnf_files_in_given_folder(self):
    self.wcnf_file_paths = glob.glob(self.dataset_folder_path + '/**/' + self._file_pattern, recursive=True)[:self.fetch_count]
    if (self.verbose > 1):
      for index, file_path in enumerate(self.wcnf_file_paths):
        print(f"Fetching  {index} th  file at : {file_path} ...")
    if (self.fetch_count != -1): self.fetch_count = min(self.fetch_count, len(self.wcnf_file_paths))

  def __start_timer(self):
    if (self._timer_lock is True):
      return
    self._timer_lock = True
    self._current_start_time = time.time()

  def __end_timer(self):
    _current_end_time = time.time()
    if (self._timer_lock is False):
      return
    self.used_time += _current_end_time - self._current_start_time
    print(f"Execution Time : {_current_end_time - self._current_start_time:.4f} seconds")
    self._current_start_time = 0
    self._timer_lock = False

  def __print_step_result(self, step_result):
    if step_result is None or step_result is False:
      print("Unsatisfiable")
    else:
      print("Best Assignment:", step_result)

  def fetch_clauses(self) -> bool:
    if (self._file_pointer >= len(self.wcnf_file_paths)):
      return False
    self._current_soft_clauses, self._current_hard_clauses \
      = self.wcnf_reader.read_wcnf_to_clauses(self.wcnf_file_paths[self._file_pointer])
    self._file_pointer = self._file_pointer + 1 if not self.fetch_random_file else random.randint(0, self.fetch_count - 1)
    self._current_soft_clauses = self.wcnf_converter.convert_to_wcnf(self._current_soft_clauses)
    self._current_hard_clauses = self.wcnf_converter.convert_to_wcnf(self._current_hard_clauses)
    return True

  def testing_model(self, solverType: SolverType, used_clause_type: int) -> bool:
    formula: WCNF
    if (used_clause_type == 0):
      formula = self._current_soft_clauses
    elif (used_clause_type == 1):
      formula = self._current_hard_clauses
    else:
      formula = self._current_soft_clauses + self._current_hard_clauses

    if (solverType not in self.limit_solvers):
      return False

    if (solverType == SolverType.FM):
      fm_solver = FM(formula)
      self.__start_timer()
      step_result = fm_solver.compute()
      self.__end_timer()
    elif (solverType == SolverType.RC2):
      rc2_solver = RC2(formula)
      self.__start_timer()
      step_result = rc2_solver.compute()
      self.__end_timer()
    elif (solverType == SolverType.LSU):
      lsu_solver = LSU(formula, verbose=1)
      self.__start_timer()
      step_result = lsu_solver.solve()
      self.__end_timer()

    if (self.verbose > 0):
      self.__print_step_result(step_result)

    return True

  def get_used_time(self):
    return self.used_time

  def get_max_iteration_count(self):
    return self.fetch_count

  def reset(self):
    self.used_time = 0
    self._file_pointer = 0

## Run the Test Steps in a Custom Loop

In [ ]:
test_step = TestStep(
  fetch_count=50,
  dataset_folder_path='/content/unweighted_maxsat_data',
  fetch_random_file=False,
  limit_solvers=[SolverType.FM, SolverType.RC2, SolverType.LSU, SolverType.BruteForce],
  verbose=0
)

In [ ]:
selected_solver = SolverType.FM
test_step.reset()

for t in range(test_step.get_max_iteration_count()):
  print(f"\nIteration {t} : ")
  if (not test_step.fetch_clauses()): break;
  test_step.testing_model(selected_solver, 0)

print(f"\nTotal Execution Time of {selected_solver} : {test_step.get_used_time():.4f} seconds")


Iteration 0 : 
Execution Time : 0.0095 seconds

Iteration 1 : 
Execution Time : 0.0192 seconds

Iteration 2 : 
Execution Time : 0.0008 seconds

Iteration 3 : 
Execution Time : 0.0002 seconds

Iteration 4 : 
Execution Time : 0.0015 seconds

Iteration 5 : 
Execution Time : 0.0002 seconds

Iteration 6 : 
Execution Time : 0.0350 seconds

Iteration 7 : 
Execution Time : 0.0700 seconds

Iteration 8 : 
Execution Time : 0.0002 seconds

Iteration 9 : 
Execution Time : 0.0010 seconds

Iteration 10 : 
Execution Time : 0.0157 seconds

Iteration 11 : 
Execution Time : 0.0073 seconds

Iteration 12 : 
Execution Time : 0.0003 seconds

Iteration 13 : 
Execution Time : 0.0009 seconds

Iteration 14 : 
Execution Time : 0.0000 seconds

Iteration 15 : 
Execution Time : 0.0270 seconds

Iteration 16 : 
Execution Time : 0.0038 seconds

Iteration 17 : 
Execution Time : 0.0027 seconds

Iteration 18 : 
Execution Time : 0.0000 seconds

Iteration 19 : 
Execution Time : 0.0010 seconds

Iteration 20 : 
Execution Tim

In [ ]:
selected_solver = SolverType.RC2
test_step.reset()

for t in range(test_step.get_max_iteration_count()):
  print(f"\nIteration {t} : ")
  test_step.fetch_clauses()
  test_step.testing_model(selected_solver, 0)
  print()

print(f"\nTotal Execution Time of {selected_solver} : {test_step.get_used_time():.4f} seconds")


Iteration 0 : 
Execution Time : 0.0431 seconds


Iteration 1 : 
Execution Time : 0.0476 seconds


Iteration 2 : 
Execution Time : 0.0057 seconds


Iteration 3 : 
Execution Time : 0.0003 seconds


Iteration 4 : 
Execution Time : 0.0081 seconds


Iteration 5 : 
Execution Time : 0.0005 seconds


Iteration 6 : 
Execution Time : 0.0757 seconds


Iteration 7 : 
Execution Time : 0.1016 seconds


Iteration 8 : 
Execution Time : 0.0002 seconds


Iteration 9 : 
Execution Time : 0.0020 seconds


Iteration 10 : 
Execution Time : 0.0255 seconds


Iteration 11 : 
Execution Time : 0.0204 seconds


Iteration 12 : 
Execution Time : 0.0008 seconds


Iteration 13 : 
Execution Time : 0.0038 seconds


Iteration 14 : 
Execution Time : 0.0002 seconds


Iteration 15 : 
Execution Time : 0.0691 seconds


Iteration 16 : 
Execution Time : 0.0141 seconds


Iteration 17 : 
Execution Time : 0.0102 seconds


Iteration 18 : 
Execution Time : 0.0001 seconds


Iteration 19 : 
Execution Time : 0.0033 seconds


Iteration

In [ ]:
selected_solver = SolverType.LSU
test_step.reset()

for t in range(test_step.get_max_iteration_count()):
  print(f"\nIteration {t} : ")
  if (not test_step.fetch_clauses()): break;
  test_step.testing_model(selected_solver, 0)

print(f"\nTotal Execution Time of {selected_solver} : {test_step.get_used_time():.4f} seconds")


Iteration 0 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0089 seconds

Iteration 1 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0095 seconds

Iteration 2 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0014 seconds

Iteration 3 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0009 seconds

Iteration 4 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0020 seconds

Iteration 5 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0010 seconds

Iteration 6 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0080 seconds

Iteration 7 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0239 seconds

Iteration 8 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0010 seconds

Iteration 9 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0017 seconds

Iteration 10 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0050 seconds

Iteration 11 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0052 seconds

Iteration 12 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0010 seconds

Iteration 13 : 
o 0
s OPTIMUM FOUND
Execution Time : 0.0019 seconds

Iteration 14 : 
s UNSATISFIABLE
Execution T

# Original source code

In [ ]:
try:
  from pysat.examples.fm import FM
  from pysat.examples.rc2 import RC2
  from pysat.examples.lsu import LSU
except:
  %pip install python-sat
  from pysat.examples.fm import FM
  from pysat.examples.rc2 import RC2
  from pysat.examples.lsu import LSU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.1 MB/s eta 0:00:00


# Decomposite LSU Algorithm

In [ ]:
#!/usr/bin/env python
#-*- coding:utf-8 -*-
##
## lsu.py
##
##  Created on: Aug 29, 2018
##      Author: Miguel Neves
##      E-mail: neves@sat.inesc-id.pt
##

"""
    ===============
    List of classes
    ===============

    .. autosummary::
        :nosignatures:

        LSU
        LSUPlus

    ==================
    Module description
    ==================

    The module implements a prototype of the known *LSU/LSUS*, e.g. *linear
    (search) SAT-UNSAT*, algorithm for MaxSAT, e.g. see [1]_.  The
    implementation is improved by the use of the *iterative totalizer encoding*
    [2]_. The encoding is used in an incremental fashion, i.e. it is created
    once and reused as many times as the number of iterations the algorithm
    makes.

    .. [1] António Morgado, Federico Heras, Mark H. Liffiton, Jordi Planes,
        Joao Marques-Silva. *Iterative and core-guided MaxSAT solving: A
        survey and assessment*. Constraints 18(4). 2013. pp. 478-534

    .. [2] Ruben Martins, Saurabh Joshi, Vasco M. Manquinho, Inês Lynce.
        *Incremental Cardinality Constraints for MaxSAT*. CP 2014. pp. 531-548

    The implementation can be used as an executable (the list of available
    command-line options can be shown using ``lsu.py -h``) in the following
    way:

    ::

        $ xzcat formula.wcnf.xz
        p wcnf 3 6 4
        1 1 0
        1 2 0
        1 3 0
        4 -1 -2 0
        4 -1 -3 0
        4 -2 -3 0

        $ lsu.py -s glucose3 -m -v formula.wcnf.xz
        c formula: 3 vars, 3 hard, 3 soft
        o 2
        s OPTIMUM FOUND
        v -1 -2 3 0
        c oracle time: 0.0000

    Alternatively, the algorithm can be accessed and invoked through the
    standard ``import`` interface of Python, e.g.

    .. code-block:: python

        >>> from pysat.examples.lsu import LSU
        >>> from pysat.formula import WCNF
        >>>
        >>> wcnf = WCNF(from_file='formula.wcnf.xz')
        >>>
        >>> lsu = LSU(wcnf, verbose=0)
        >>> lsu.solve()  # set of hard clauses should be satisfiable
        True
        >>> print(lsu.cost) # cost of MaxSAT solution should be 2
        >>> 2
        >>> print(lsu.model)
        [-1, -2, 3]

    ==============
    Module details
    ==============
"""

#
#==============================================================================
from __future__ import print_function
import getopt
from pysat.card import ITotalizer
from pysat.formula import CNF, WCNF, WCNFPlus
from pysat.solvers import Solver, SolverNames
from threading import Timer
import os
import sys
import re


# TODO: support weighted MaxSAT
#==============================================================================
class LSU:
    """
        Linear SAT-UNSAT algorithm for MaxSAT [1]_. The algorithm can be seen
        as a series of satisfiability oracle calls refining an upper bound on
        the MaxSAT cost, followed by one unsatisfiability call, which stops the
        algorithm. The implementation encodes the sum of all selector literals
        using the *iterative totalizer encoding* [2]_. At every iteration, the
        upper bound on the cost is reduced and enforced by adding the
        corresponding unit size clause to the working formula. No clauses are
        removed during the execution of the algorithm. As a result, the SAT
        oracle is used incrementally.

        .. warning:: At this point, :class:`LSU` supports only
            **unweighted** problems.

        The constructor receives an input :class:`.WCNF` formula, a name of the
        SAT solver to use (see :class:`.SolverNames` for details), and an
        integer verbosity level.

        :param formula: input MaxSAT formula
        :param solver: name of SAT solver
        :param incr: enable incremental mode of Glucose
        :param expect_interrupt: whether or not an :meth:`interrupt` call is expected
        :param verbose: verbosity level

        :type formula: :class:`.WCNF`
        :type solver: str
        :type incr: bool
        :type expect_interrupt: bool
        :type verbose: int
    """

    def __init__(self, formula, solver='g4', incr=False, expect_interrupt=False, verbose=0):
        """
            Constructor.
        """

        self.verbose = verbose
        self.solver = solver
        self.incr = incr
        self.expect_interrupt = expect_interrupt
        self.formula = formula
        self.topv = formula.nv  # largest variable index
        self.sels = []          # soft clause selector variables
        self.tot = None         # totalizer encoder for the cardinality constraint
        self._init(formula)     # initiaize SAT oracle

    def _init(self, formula):
        """
            SAT oracle initialization. The method creates a new SAT oracle and
            feeds it with the formula's hard clauses. Afterwards, all soft
            clauses of the formula are augmented with selector literals and
            also added to the solver. The list of all introduced selectors is
            stored in variable ``self.sels``.

            :param formula: input MaxSAT formula
            :type formula: :class:`WCNF`
        """

        self.oracle = Solver(name=self.solver, bootstrap_with=formula.hard, incr=self.incr, use_timer=True)

        for i, cl in enumerate(formula.soft):
            # TODO: if clause is unit, use its literal as selector
            # (ITotalizer must be extended to support PB constraints first)
            self.topv += 1
            selv = self.topv
            cl.append(self.topv)
            self.oracle.add_clause(cl)
            self.sels.append(selv)

        if self.verbose > 1:
            print('c formula: {0} vars, {1} hard, {2} soft'.format(formula.nv, len(formula.hard), len(formula.soft)))

    def __del__(self):
        """
            Destructor.
        """

        self.delete()

    def __enter__(self):
        """
            'with' constructor.
        """

        return self

    def __exit__(self, exc_type, exc_value, traceback):
        """
            'with' destructor.
        """

        self.delete()

    def delete(self):
        """
            Explicit destructor of the internal SAT oracle and the
            :class:`.ITotalizer` object.
        """

        if self.oracle:
            self.oracle.delete()
            self.oracle = None

        if self.tot:
            self.tot.delete()
            self.tot = None

    def solve(self):
        """
            Computes a solution to the MaxSAT problem. The method implements
            the LSU/LSUS algorithm, i.e. it represents a loop, each iteration
            of which calls a SAT oracle on the working MaxSAT formula and
            refines the upper bound on the MaxSAT cost until the formula
            becomes unsatisfiable.

            Returns ``True`` if the hard part of the MaxSAT formula is
            satisfiable, i.e. if there is a MaxSAT solution, and ``False``
            otherwise.

            :rtype: bool
        """

        is_sat = False

        while self.oracle.solve_limited(expect_interrupt=self.expect_interrupt):
            is_sat = True
            self.model = self.oracle.get_model()
            self.cost = self._get_model_cost(self.formula, self.model)
            if self.verbose:
                print('o {0}'.format(self.cost))
                sys.stdout.flush()
            if self.cost == 0:      # if cost is 0, then model is an optimum solution
                break
            self._assert_lt(self.cost)

        if is_sat:
            self.model = filter(lambda l: abs(l) <= self.formula.nv, self.model)
            if self.verbose:
                if self.found_optimum():
                    print('s OPTIMUM FOUND')
                else:
                    print('s SATISFIABLE')
        elif self.verbose:
            print('s UNSATISFIABLE')

        return is_sat

    def get_model(self):
        """
            This method returns a model obtained during a prior satisfiability
            oracle call made in :func:`solve`.

            :rtype: list(int)
        """

        return self.model

    def found_optimum(self):
        """
            Checks if the optimum solution was found in a prior call to
            :func:`solve`.

            :rtype: bool
        """

        return self.oracle.get_status() is not None

    def _get_model_cost(self, formula, model):
        """
            Given a WCNF formula and a model, the method computes the MaxSAT
            cost of the model, i.e. the sum of weights of soft clauses that are
            unsatisfied by the model.

            :param formula: an input MaxSAT formula
            :param model: a satisfying assignment

            :type formula: :class:`.WCNF`
            :type model: list(int)

            :rtype: int
        """

        model_set = set(model)
        cost = 0

        for i, cl in enumerate(formula.soft):
            cost += formula.wght[i] if all(l not in model_set for l in filter(lambda l: abs(l) <= self.formula.nv, cl)) else 0

        return cost

    def _assert_lt(self, cost):
        """
            The method enforces an upper bound on the cost of the MaxSAT
            solution. This is done by encoding the sum of all soft clause
            selectors with the use the iterative totalizer encoding, i.e.
            :class:`.ITotalizer`. Note that the sum is created once, at the
            beginning. Each of the following calls to this method only enforces
            the upper bound on the created sum by adding the corresponding unit
            size clause. Each such clause is added on the fly with no restart
            of the underlying SAT oracle.

            :param cost: the cost of the next MaxSAT solution is enforced to be
                *lower* than this current cost

            :type cost: int
        """

        if self.tot == None:
            self.tot = ITotalizer(lits=self.sels, ubound=cost-1, top_id=self.topv)
            self.topv = self.tot.top_id

            for cl in self.tot.cnf.clauses:
                self.oracle.add_clause(cl)

        self.oracle.add_clause([-self.tot.rhs[cost-1]])

    def interrupt(self):
        """
            Interrupt the current execution of LSU's :meth:`solve` method.
            Can be used to enforce time limits using timer objects. The
            interrupt must be cleared before running the LSU algorithm again
            (see :meth:`clear_interrupt`).
        """

        self.oracle.interrupt()

    def clear_interrupt(self):
        """
            Clears an interruption.
        """

        self.oracle.clear_interrupt()

    def oracle_time(self):
        """
            Method for calculating and reporting the total SAT solving time.
        """

        return self.oracle.time_accum()


#
#==============================================================================
class LSUPlus(LSU, object):
    """
        LSU-like algorithm extended for :class:`.WCNFPlus` formulas (using
        :class:`.Minicard`).

        :param formula: input MaxSAT formula in WCNF+ format
        :param expect_interrupt: whether or not an :meth:`interrupt` call is expected
        :param verbose: verbosity level

        :type formula: :class:`.WCNFPlus`
        :type expect_interrupt: bool
        :type verbose: int
    """

    def __init__(self, formula, solver='g4', incr=False, expect_interrupt=False, verbose=0):
        """
            Constructor.
        """

        assert solver in SolverNames.gluecard3 or \
               solver in SolverNames.gluecard4 or \
               solver in SolverNames.minicard or \
               solver in SolverNames.cadical195, '{0} does not support native cardinality constraints'.format(solver)

        super(LSUPlus, self).__init__(formula, solver=solver, incr=incr,
                expect_interrupt=expect_interrupt, verbose=verbose)

        # we are using CaDiCaL195 and it can use external linear engine
        if solver in SolverNames.cadical195:
            self.oracle.activate_atmost()

        # adding atmost constraints
        for am in formula.atms:
            self.oracle.add_atmost(*am)

    def _assert_lt(self, cost):
        """
            Overrides _assert_lt of :class:`.LSU` in order to use Minicard's
            native support for cardinality constraints

            :param cost: the cost of the next MaxSAT solution is enforced to
                be *lower* than this current cost

            :type cost: int
        """

        self.oracle.add_atmost(self.sels, cost-1)
